# Intro to ML and Linear Regression

Based on the **Google ML Crash Course**, this notebook walks through the fundamentals of Linear Regression — the simplest and most foundational supervised learning algorithm.

---

## What is Linear Regression?

**Linear Regression** is a statistical method used to model the relationship between a **feature** (input) and a **label** (output) by fitting a straight line to the observed data.

### The Equation of a Line

$$y = wx + b$$

| Symbol | ML Term | Meaning |
|--------|---------|----------|
| `y`    | **Label** | The value we want to predict (e.g., taxi fare) |
| `x`    | **Feature** | The input variable used to make the prediction (e.g., trip miles) |
| `w`    | **Weight** | The slope of the line — how much `y` changes per unit of `x` |
| `b`    | **Bias** | The y-intercept — the value of `y` when `x = 0` |

---

## Key Terms

- 📊 **Features**: The input variables (columns) that the model uses to make predictions.
- 🏷️ **Labels**: The output variable the model is trained to predict.
- ⚖️ **Weights (`w`)**: Learned parameters that determine how strongly each feature influences the prediction.
- ➕ **Bias (`b`)**: A learned offset that allows the model to fit data that doesn't pass through the origin.
- 📉 **Mean Squared Error (MSE)**: The loss function used to measure how wrong the model's predictions are. It is the average of the squared differences between predictions and actual values.
- 🔄 **Epochs**: One complete pass through the entire training dataset. More epochs = more opportunities to learn.
- 🎓 **Learning Rate**: A hyperparameter that controls how big of a step the model takes when adjusting weights during training. Too high → unstable. Too low → slow to learn.

In [ ]:
# ──────────────────────────────────────────────────────
# Setup and Data Loading
# ──────────────────────────────────────────────────────
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

print(f"TensorFlow version: {tf.__version__}")

# Load the Chicago Taxi dataset from the Google MLCC servers
DATA_URL = 'https://download.mlcc.google.com/mledu-datasets/chicago_taxi_train.csv'
df = pd.read_csv(DATA_URL)

# Define columns of interest
FEATURE = 'TRIP_MILES'   # Input: How far the taxi traveled
LABEL   = 'FARE'         # Output: How much the rider paid

# Keep only the columns we need and drop any missing values
df = df[[FEATURE, LABEL]].dropna()

print(f"Dataset loaded: {df.shape[0]:,} rows")
print(df.head())

# Exploratory Data Analysis

Before building any model, it's important to **visualize the data**.

By plotting `TRIP_MILES` (how far the taxi traveled) against `FARE` (how much was charged), we can immediately see:
- Whether a **linear relationship** exists between the two variables.
- The **spread** and any **outliers** in the data.

A clear upward trend in the scatter plot would confirm that linear regression is a suitable model.

In [ ]:
# ──────────────────────────────────────────────────────
# Visualizing the Data — Scatter Plot
# ──────────────────────────────────────────────────────
plt.figure(figsize=(10, 6))

plt.scatter(
    df[FEATURE],
    df[LABEL],
    alpha=0.2,        # Transparency to show density
    s=10,             # Point size
    color='steelblue',
    label='Actual data'
)

plt.xlabel('Trip Miles', fontsize=13)
plt.ylabel('Fare (USD)', fontsize=13)
plt.title('Chicago Taxi: Trip Miles vs. Fare', fontsize=15)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

# Building and Training the Model

We use **`tf.keras.Sequential`** to build a simple linear regression model.

The model has just **one layer** — a `Dense` layer with a single neuron. This directly implements the line equation:

$$\text{FARE} = w \times \text{TRIP\_MILES} + b$$

- **Optimizer**: `RMSprop` — an adaptive learning rate optimizer that works well for regression tasks.
- **Loss**: `Mean Squared Error (MSE)` — penalizes large prediction errors more than small ones.
- **Epochs**: The number of times the model will see the entire training dataset.
- **Batch Size**: The number of samples used per gradient update.

In [ ]:
# ──────────────────────────────────────────────────────
# Model Definition, Compilation, and Training
# ──────────────────────────────────────────────────────

# Hyperparameters
LEARNING_RATE = 0.001
EPOCHS        = 50
BATCH_SIZE    = 100

# Define the model: one Dense layer = linear regression
model = tf.keras.Sequential([
    tf.keras.layers.Dense(
        units=1,          # One output neuron (the predicted fare)
        input_shape=(1,)  # One input feature (trip miles)
    )
])

# Compile the model
model.compile(
    optimizer=tf.keras.optimizers.RMSprop(learning_rate=LEARNING_RATE),
    loss='mean_squared_error',
    metrics=['mean_absolute_error']
)

model.summary()

# Train the model
history = model.fit(
    df[FEATURE],
    df[LABEL],
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=1
)

print("\nTraining complete!")

# Evaluating the Model

After training, we can inspect the **learned parameters** directly from the model's layer:

- **Weight (`w`)**: The learned fare rate per mile.
- **Bias (`b`)**: The base fare at zero miles (analogous to a taxi's flag-fall or booking fee).

We then plot the **regression line** over the original scatter plot to visually assess the quality of the model's fit.

In [ ]:
# ──────────────────────────────────────────────────────
# Extract Learned Parameters and Plot the Regression Line
# ──────────────────────────────────────────────────────
import numpy as np

# Extract the learned weight (slope) and bias (intercept)
trained_layer = model.layers[0]
weight = trained_layer.get_weights()[0][0][0]   # w
bias   = trained_layer.get_weights()[1][0]       # b

print(f"Learned weight (fare per mile): ${weight:.4f}")
print(f"Learned bias   (base fare):     ${bias:.4f}")
print(f"\nEquation: FARE = {weight:.2f} × TRIP_MILES + {bias:.2f}")

# Generate x-values for the regression line
x_line = np.linspace(df[FEATURE].min(), df[FEATURE].max(), 200)
y_line = weight * x_line + bias

# Plot scatter + regression line
plt.figure(figsize=(10, 6))

plt.scatter(
    df[FEATURE],
    df[LABEL],
    alpha=0.2,
    s=10,
    color='steelblue',
    label='Actual data'
)

plt.plot(
    x_line,
    y_line,
    color='crimson',
    linewidth=2.5,
    label=f'Regression line: FARE = {weight:.2f}×miles + {bias:.2f}'
)

plt.xlabel('Trip Miles', fontsize=13)
plt.ylabel('Fare (USD)', fontsize=13)
plt.title('Chicago Taxi: Linear Regression Fit', fontsize=15)
plt.legend(fontsize=11)
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()